# TD 7 — L'équation de Mincer

**Analyse des données — L3 Économie**

Combien rapporte une année d'études supplémentaire ? La régression qui répond à cette question — posée par Jacob Mincer en 1974 — est la plus estimée de toute l'économie appliquée.

Vous allez l'estimer. Mais l'exercice porte surtout sur une compétence : **dire en une phrase** ce que signifie chaque coefficient.

> *Exécution → Tout exécuter* avant de démarrer.


## Partie 1 — Préparer les variables

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm03-salaires.csv")

df = pd.read_csv(URL, sep=";", encoding="utf-8", decimal=",", thousands=" ")
print(df.shape)
df.head(3)

**Question 1.** Appliquez le nettoyage de la séance 3, puis comptez les lignes restantes.

In [ ]:
# Le nettoyage de la seance 3
df["SALAIRE_NET"] = df["SALAIRE_NET"].replace(999999, np.nan)
df["DIPLOME"]     = df["DIPLOME"].replace(9, np.nan)
df["TEMPS"]       = df["TEMPS"].replace(9, np.nan)

df = df[df["SALAIRE_NET"] > 0].copy()      # ecarte les non-salaries ET les NaN
print(len(df), "lignes avec un salaire renseigne et non nul")

**Question 2.** Pourquoi régresse-t-on le **logarithme** du salaire plutôt que le salaire ? Deux raisons : l'une sur la forme de la distribution, l'autre sur la lecture des coefficients.

*Votre réponse :*


In [ ]:
# Un argument se montre. Comparez les deux distributions.
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(df["SALAIRE_NET"], bins=50); ax[0].set_title("salaire")
ax[1].hist(np.log(df["SALAIRE_NET"]), bins=50); ax[1].set_title("log(salaire)")
plt.show()

**Question 3.** Traduisez `DIPLOME` en années d'études, construisez l'expérience potentielle et la muette `FEMME`.

En quoi la table de conversion est-elle une **hypothèse**, et non une simple traduction ?

*Votre réponse :*


In [ ]:
ANNEES = {1: 7, 2: 9, 3: 11, 4: 12, 5: 14, 6: 15, 7: 17}

df["ETUDE"] = df["DIPLOME"].map(ANNEES)
df["EXPER"] = df["AGE"] - df["ETUDE"] - 6
df["FEMME"] = (df["SEXE"] == 2).astype(int)
df["LSAL"]  = np.log(df["SALAIRE_NET"])

d = df.dropna(subset=["LSAL", "ETUDE", "EXPER"]).copy()
print(len(d), "lignes avec salaire et diplome")
print((d["EXPER"] < 0).sum(), "experiences negatives")
d = d[d["EXPER"] >= 0].copy()
print(len(d), "lignes dans l'echantillon final")

**Question 4.** Combien d'expériences négatives ? Que faites-vous de ces lignes, et que coûte votre décision ?

**Question 5.** Pourquoi une seule colonne suffit-elle à coder deux sexes ?

*Vos réponses :*


## Partie 2 — Une différence de moyennes est une régression

**Question 6.** Moyenne de `LSAL` chez les hommes, chez les femmes, et leur différence.

In [ ]:
# a vous

**Question 7.** Estimez `LSAL ~ FEMME`. Comparez la constante et le coefficient aux deux nombres précédents.

In [ ]:
m0 = smf.ols("LSAL ~ FEMME", data=d).fit(cov_type="HC1")
print(m0.summary().tables[1])

**Question 8.** Refaites le test de Welch de la séance 5, et comparez son $t$ à celui du coefficient.

*Votre réponse :*


In [ ]:
from scipy import stats
# a vous

## Partie 3 — L'équation de Mincer

**Question 9.** `LSAL ~ ETUDE`, puis la même chose avec l'expérience et son carré. Le rendement change-t-il ?

In [ ]:
m1 = smf.ols("LSAL ~ ETUDE", data=d).fit(cov_type="HC1")
m2 = smf.ols("LSAL ~ ETUDE + EXPER + I(EXPER**2)", data=d).fit(cov_type="HC1")

print(m1.params.round(5).to_string())
print()
print(m2.params.round(5).to_string())

**Question 10.** Le coefficient du carré est négatif. Que décrit-il ? À combien d'années le profil atteint-il son sommet ?

*Votre réponse :*


In [ ]:
# le sommet d'une parabole b1*x + b2*x^2 est en -b1 / (2*b2)

**Question 11.** Ajoutez `FEMME`. Écrivez **une phrase par coefficient**, constante comprise.

*Vos quatre phrases :*


In [ ]:
m3 = smf.ols("LSAL ~ ETUDE + EXPER + I(EXPER**2) + FEMME",
             data=d).fit(cov_type="HC1")
print(m3.summary())

## Partie 4 — L'interaction

**Question 12.** Ajoutez `ETUDE:FEMME`. Quelle hypothèse ce coefficient teste-t-il ? Est-il significatif ? Et que signifie désormais le coefficient de `FEMME` seul ?

*Votre réponse :*


In [ ]:
m4 = smf.ols("LSAL ~ ETUDE + EXPER + I(EXPER**2) + FEMME + ETUDE:FEMME",
             data=d).fit(cov_type="HC1")
print(m4.summary().tables[1])

**Question 13.** Ajoutez le temps de travail. Que devient l'écart femmes-hommes ? Rapprochez-le des 336 € et 146 € de la séance 5.

*Votre réponse :*


In [ ]:
m5 = smf.ols("LSAL ~ ETUDE + EXPER + I(EXPER**2) + FEMME + ETUDE:FEMME + C(TEMPS)",
             data=d).fit(cov_type="HC1")
print(m5.summary().tables[1])

# qui travaille a temps partiel ?
print(pd.crosstab(d["FEMME"], d["TEMPS"], normalize="index").round(3))

## Partie 5 — Les écarts-types

**Question 14.** Comparez écarts-types simples et robustes, en logarithme puis en niveau.

*Votre réponse :*


In [ ]:
for formule in ["LSAL ~ ETUDE", "SALAIRE_NET ~ ETUDE"]:
    a = smf.ols(formule, data=d).fit()                    # simples
    b = smf.ols(formule, data=d).fit(cov_type="HC1")      # robustes
    print(f"{formule:<22} coef {a.params['ETUDE']:9.4f}"
          f"   se {a.bse['ETUDE']:8.4f}   seHC1 {b.bse['ETUDE']:8.4f}")

**Question 15.** Lequel de ces modèles publieriez-vous ? Que votre coefficient d'éducation **ne mesure pas** ?

*Votre réponse :*


## Pour ceux qui ont terminé

**Question 16.** Remplacez les années d'études par `C(DIPLOME)`. Les écarts entre niveaux sont-ils réguliers ?

**Question 17.** Estimez séparément sur les hommes et sur les femmes. Retrouvez-vous l'interaction ?

**Question 18.** Trois déterminants du salaire absents du fichier, et le sens du biais pour chacun.


In [ ]:
# a vous